# Step 3. SUNA nitrate quality control (L3)

Implements the post-processing quality control of Zheng et al. (2024) for the rosette-mounted
SUNA, producing the L3 quality-controlled nitrate product.

```text
Inputs
  *_SUNA_1s.csv    <- L2/SUNA     (merged CTD+SUNA @1s, from step02; carries raw spectra if present)
  raw SUNA .csv    <- L1/SUNA     (full SATSLF frames with 256-channel spectra)
  <serial>.cal     <- calibration (per-wavelength eps_NO3, bromide SWA/TSWA, reference spectrum)
  bottle nitrate   <- discrete samples (template schema below)
Output
  *_nitrate_QC.csv -> L3
  assessment + audit -> _audit
```

## The three Zheng correction stages (their Fig. 3)

1. **TSP spectral correction** (Sakamoto 2009/2017b; Plant 2023). Re-derives nitrate from the raw
   256-channel UV absorbance over 217–240 nm, removing the bromide (sea-salt) absorbance using CTD
   temperature/salinity and the cal coefficients, then least-squares fitting the residual to a linear
   baseline + nitrate extinction. **Buildable now** — needs only raw spectra + cal + CTD T/S/P.
2. **Low-nitrate temperature-dependent residual.** For bottle match-ups < 2.5 µM, regress the
   (SUNA − bottle) residual against temperature to obtain c1, c2, then subtract c1·T + c2 where nitrate
   < 2.5 µM. Coefficients are **instrument-specific and derived from your own data**, not Zheng's.
3. **Cruise-specific full-range bias.** Regress the corrected SUNA against all bottle nitrate for the
   cruise to obtain slope a1 and intercept a0, then apply N_QC = (N − a0)/a1.

Stages 2–3 require a bottle match-up table. They are written to consume the template produced in
Section 2 and run with no code change once that table is filled and keyed to casts/depths.

## Important data-provenance notes for this build

* The pipeline is **cruise-agnostic**: it loads one `CRUISE_ID`, its merged SUNA, and only that
  cruise's bottle rows. It will not mix cruises — Zheng's bias correction is per-cruise because the
  bias drifts between deployments. Do not calibrate one cruise's SUNA against another's bottles.
* The summer surface zero-clamp from Zheng (force N_QC=0 in the depleted surface layer) is tuned to
  the Northeast U.S. Shelf and is **off by default**; the Gulf of Guinea upwelling regime may not be
  nitrate-depleted at the surface. Enable only with local justification.


## 1. Imports and settings

In [1]:
from __future__ import annotations

import re
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


In [2]:
# ===========================================================================
# Paths (match step01/step02 v2 layout)
# ===========================================================================
CTD_ROOT     = Path(r"C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd")
CRUISES_ROOT = CTD_ROOT / "cruises"

CRUISE_ID  = "P45_06"            # set to the cruise being QC'd; bottles must match
CRUISE_DIR = CRUISES_ROOT / CRUISE_ID

L1_SUNA_ROOT = CRUISE_DIR / "L1" / "SUNA"          # raw SATSLF .csv frames
L2_SUNA_ROOT = CRUISE_DIR / "L2" / "SUNA"          # merged *_SUNA_1s.csv (step02)
L3_ROOT      = CRUISE_DIR / "L3"                   # QC nitrate output
AUDIT_ROOT   = CRUISE_DIR / "_audit"
BOTTLE_DIR   = CRUISE_DIR / "metadata" / "bottle_nitrate"  # bottle table + template

for d in (L3_ROOT, AUDIT_ROOT, BOTTLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Calibration file (per-wavelength eps_NO3, SWA, TSWA, Reference)
CAL_FILE = CTD_ROOT / "metadata" / "calibration" / "SNA2503C.cal"

# SUNA cast->file map (same file step02 used) so each cast reads its correct raw SUNA log.
SUNA_MAP_FILE = CRUISE_DIR / "metadata" / "suna_cast_map.csv"

# Bottle match-up table (filled from the template in Section 2)
BOTTLE_TABLE = BOTTLE_DIR / f"{CRUISE_ID}_bottle_nitrate.csv"

# ===========================================================================
# TSP fit window and stage thresholds (Zheng 2024)
# ===========================================================================
FIT_WL_MIN, FIT_WL_MAX = 217.0, 240.0   # nitrate absorption window (nm)
LOW_NITRATE_THRESHOLD  = 2.5            # uM; Stage 2 applies below this
ABSORBANCE_CUTOFF      = 1.30           # from SUNA header; ignore saturated channels

# Stage 0 match-up windows
MATCH_DEPTH_TOL_M  = 1.0    # bottle<->SUNA depth tolerance (m)
MATCH_TIME_TOL_S   = 30.0   # if bottle time present, +/- window for averaging SUNA scans
REPLICATE_MAX_DIFF = 0.5    # uM; exclude replicate pairs differing by more (Zheng)

# Stage 3 guards
MIN_BOTTLES_FOR_FIT = 10    # warn below this; Zheng targets ~50
APPLY_SUMMER_SURFACE_ZERO = False   # NES-specific; off for Gulf of Guinea

# Which nitrate to start from in Stage 1.
#   "onboard" -> use the SUNA onboard nitrate (firmware TSP) as Stage-1 start [DEFAULT, validated]
#   "tsp"     -> re-derive nitrate from raw spectra. WORK IN PROGRESS - see tsp_correct() note.
STAGE1_MODE = "onboard"

print("CRUISE_ID :", CRUISE_ID)
print("CAL_FILE  :", CAL_FILE, "(exists:", CAL_FILE.exists(), ")")
print("BOTTLE    :", BOTTLE_TABLE, "(exists:", BOTTLE_TABLE.exists(), ")")
print("STAGE1    :", STAGE1_MODE)


CRUISE_ID : P45_06
CAL_FILE  : C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\metadata\calibration\SNA2503C.cal (exists: True )
BOTTLE    : C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\metadata\bottle_nitrate\P45_06_bottle_nitrate.csv (exists: False )
STAGE1    : onboard


## 2. Bottle match-up template

Stages 2–3 need a bottle table with explicit numeric depth (and ideally time), keyed to casts.
The qualitative SURF/ML/DCM labels in the raw WHOI report cannot place a bottle at a metre, so this
cell writes a ready-to-fill template with the required schema. Fill it (one row per replicate),
save as `<CRUISE_ID>_bottle_nitrate.csv` in the bottle folder, and rerun from Section 6.

Below-detection values go in `flag` (e.g. `BDL`), not in `nitrate_uM`, so they are not parsed as
numbers and can be handled explicitly.

In [3]:
BOTTLE_SCHEMA = {
    "cruise_id":   "e.g. P45_05",
    "cast_id":     "e.g. P45_05_CTD_03 (must match a SUNA cast)",
    "station":     "optional, e.g. J2",
    "depth_m":     "Niskin firing depth in metres (REQUIRED, numeric)",
    "time_utc":    "optional ISO datetime of firing; enables time+depth matching",
    "nitrate_uM":  "NO3+NO2 in uM (numeric; leave blank if below detection)",
    "replicate_id":"optional; same value for replicates of one sample",
    "flag":        "optional: BDL (below detection), NES (not enough sample), etc.",
}

template_path = BOTTLE_DIR / f"{CRUISE_ID}_bottle_nitrate_TEMPLATE.csv"
if not template_path.exists():
    tmpl = pd.DataFrame(columns=list(BOTTLE_SCHEMA.keys()))
    # write a commented schema row then an empty example row
    tmpl.loc[0] = list(BOTTLE_SCHEMA.values())
    tmpl.to_csv(template_path, index=False, encoding="utf-8-sig")
    print(f"Template written: {template_path}")
    print("Fill it, save as the BOTTLE_TABLE name, then rerun from Section 6.")
else:
    print(f"Template already present: {template_path}")

display(pd.DataFrame({"column": list(BOTTLE_SCHEMA.keys()),
                      "meaning": list(BOTTLE_SCHEMA.values())}))


Template written: C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\ctd\cruises\P45_06\metadata\bottle_nitrate\P45_06_bottle_nitrate_TEMPLATE.csv
Fill it, save as the BOTTLE_TABLE name, then rerun from Section 6.


,column,meaning
0,cruise_id,e.g. P45_05
1,cast_id,e.g. P45_05_CTD_03 (must match a SUNA cast)
2,station,"optional, e.g. J2"
3,depth_m,"Niskin firing depth in metres (REQUIRED, numeric)"
4,time_utc,optional ISO datetime of firing; enables time+...
5,nitrate_uM,NO3+NO2 in uM (numeric; leave blank if below d...
6,replicate_id,optional; same value for replicates of one sample
7,flag,"optional: BDL (below detection), NES (not enou..."


## 3. Calibration reader

Parses the SUNA `.cal` file: the `H,` header parameters (T_CAL, path length, fit window) and the
256 `E,` lines giving per-wavelength nitrate extinction (`eps_NO3`), sea-water bromide absorption
(`SWA`), its temperature coefficient (`TSWA`), and the reference intensity spectrum.

In [4]:
def read_cal(path: Path) -> Dict[str, Any]:
    """Read a SUNA .cal file into header params + per-wavelength arrays."""
    text = path.read_text(encoding="latin-1", errors="replace").splitlines()
    params: Dict[str, Any] = {}
    rows: List[Tuple[float, float, float, float, float]] = []

    for line in text:
        parts = [p.strip() for p in line.split(",")]
        if not parts:
            continue
        if parts[0] == "H":
            body = ",".join(parts[1:])
            for key in ("T_CAL_SWA", "T_CAL", "PATH_LENGTH", "INT_PERIOD",
                        "CONC_CAL_NO3", "CONC_CAL_SWA"):
                m = re.search(rf"{key}\s+([-\d.]+)", body)
                if m:
                    params[key] = float(m.group(1))
        elif parts[0] == "E" and len(parts) >= 6:
            try:
                wl, eps, swa, tswa, ref = (float(parts[1]), float(parts[2]),
                                           float(parts[3]), float(parts[4]), float(parts[5]))
                rows.append((wl, eps, swa, tswa, ref))
            except ValueError:
                continue

    arr = np.array(rows, dtype=float)
    cal = {
        "params": params,
        "wavelength": arr[:, 0],
        "eps_no3":    arr[:, 1],   # nitrate extinction coefficient per wavelength
        "swa":        arr[:, 2],   # sea-water (bromide) absorption at T_CAL_SWA
        "tswa":       arr[:, 3],   # temperature sensitivity of bromide absorption
        "reference":  arr[:, 4],   # reference (no-nitrate) intensity spectrum, counts
        "T_CAL":      params.get("T_CAL", 20.0),
        "T_CAL_SWA":  params.get("T_CAL_SWA", params.get("T_CAL", 20.0)),
        "path_cm":    params.get("PATH_LENGTH", 10.0) / 10.0,  # mm -> cm
    }
    return cal


cal = read_cal(CAL_FILE)
print("cal channels:", len(cal["wavelength"]),
      " wl span:", round(cal["wavelength"].min(), 1), "-", round(cal["wavelength"].max(), 1))
print("T_CAL:", cal["T_CAL"], " T_CAL_SWA:", cal["T_CAL_SWA"], " path(cm):", cal["path_cm"])
in_win = (cal["wavelength"] >= FIT_WL_MIN) & (cal["wavelength"] <= FIT_WL_MAX)
print("channels in fit window:", int(in_win.sum()))


cal channels: 256  wl span: 190.4 - 393.9
T_CAL: 20.0  T_CAL_SWA: 20.0  path(cm): 1.0
channels in fit window: 29


## 4. Raw SUNA frame reader

Parses the `SATSLF` full-ASCII frames: timestamp (year+day-of-year and decimal hour), onboard
nitrate, dark value, and the 256-channel intensity spectrum. Field positions follow the SUNA-V2
full-ASCII layout verified against this instrument (SN 2503): spectrum occupies fields 12–267
(0-based 11–266), with the per-frame dark value at field 10.

In [5]:
# Field layout (1-based) for SATSLF frames, SUNA-V2 full ASCII, SN2503.
SUNA_F_DATE       = 2     # YYYYDDD (year + day of year)
SUNA_F_HOUR       = 3     # decimal hour UTC
SUNA_F_NO3_UM     = 4     # onboard nitrate, uM
SUNA_F_DARK       = 10    # dark value (counts) used for this frame
SUNA_F_SPEC_START = 12    # first spectral channel
SUNA_N_CHANNELS   = 256


def _yeardoy_hour_to_utc(yeardoy: float, hour: float) -> Optional[pd.Timestamp]:
    try:
        s = str(int(yeardoy))
        year = int(s[:4]); doy = int(s[4:])
        base = pd.Timestamp(year=year, month=1, day=1, tz="UTC") + pd.Timedelta(days=doy - 1)
        return base + pd.Timedelta(seconds=float(hour) * 3600.0)
    except Exception:
        return None


def read_suna_frames(path: Path) -> pd.DataFrame:
    """Read raw SATSLF frames -> DataFrame with utc_time, onboard nitrate, dark, spectrum block."""
    recs = []
    spec_cols = [f"ch{j:03d}" for j in range(SUNA_N_CHANNELS)]
    for raw in path.read_text(encoding="latin-1", errors="replace").splitlines():
        if not raw.startswith("SATSLF"):
            continue
        f = raw.strip().split(",")
        need = SUNA_F_SPEC_START - 1 + SUNA_N_CHANNELS
        if len(f) < need:
            continue
        try:
            t = _yeardoy_hour_to_utc(float(f[SUNA_F_DATE - 1]), float(f[SUNA_F_HOUR - 1]))
            no3 = float(f[SUNA_F_NO3_UM - 1])
            dark = float(f[SUNA_F_DARK - 1])
            spec = [float(f[SUNA_F_SPEC_START - 1 + j]) for j in range(SUNA_N_CHANNELS)]
        except ValueError:
            continue
        recs.append([t, no3, dark] + spec)

    df = pd.DataFrame(recs, columns=["utc_time", "no3_onboard", "dark"] + spec_cols)
    df = df.dropna(subset=["utc_time"]).sort_values("utc_time").reset_index(drop=True)
    return df, spec_cols


## 5. Stage 1 — TSP spectral correction

Following Sakamoto et al. (2009, 2017b) and Plant et al. (2023):

1. Absorbance per wavelength: `A(λ) = −log10((I(λ) − dark) / Reference(λ))`.
2. Estimate the bromide (sea-salt) absorbance from CTD salinity and temperature:
   `A_swa(λ) = S · [SWA(λ) + TSWA(λ)·(T − T_CAL_SWA)] / CONC_CAL_SWA`, and subtract it.
3. Over 217–240 nm, least-squares fit the bromide-subtracted absorbance to a linear baseline plus
   nitrate extinction: `A_resid(λ) = e + f·λ + eps_NO3(λ)·c_NO3`, solving for c_NO3.

Pressure has a small effect on the bromide term; for a shelf cast it is minor and folded into the
salinity scaling here. The pressure refinement of Sakamoto (2017b) can be added if deep casts need
it. Requires CTD temperature and salinity time-matched to each SUNA frame (from step02).

In [6]:
def tsp_correct(frames: pd.DataFrame, spec_cols: List[str], cal: Dict[str, Any],
                temp_c: np.ndarray, salinity: np.ndarray) -> pd.DataFrame:
    """Re-derive nitrate from raw spectra via TSP correction. Returns frames + 'no3_tsp'."""
    wl   = cal["wavelength"]
    eps  = cal["eps_no3"]
    swa  = cal["swa"]
    tswa = cal["tswa"]
    ref  = cal["reference"]
    conc_swa = cal["params"].get("CONC_CAL_SWA", 34.91)
    t_cal_swa = cal["T_CAL_SWA"]

    # NOTE (WORK IN PROGRESS): this fit reproduces the *structure* of the Sakamoto/Plant
    # TSP correction but is not yet numerically validated against the onboard nitrate for
    # this instrument's narrow 217-240 nm window. The Bio-Argo DAC manual specifies a
    # baseline built around an OPTICAL_WAVELENGTH_OFFSET (~210 nm; Sakamoto 2009) rather
    # than a raw-wavelength linear term; without it the nitrate column is near-collinear
    # with the baseline and the fit can be unstable. Until that offset parameterization is
    # implemented and validated (target: reproduce onboard NO3 to ~1 uM), use
    # STAGE1_MODE="onboard". Centering the wavelength below is a partial conditioning fix.
    in_win = (wl >= FIT_WL_MIN) & (wl <= FIT_WL_MAX)
    wl_w, eps_w, swa_w, tswa_w, ref_w = (wl[in_win], eps[in_win], swa[in_win],
                                         tswa[in_win], ref[in_win])
    wl_ref = 210.0  # OPTICAL_WAVELENGTH_OFFSET (Sakamoto 2009); baseline reference

    spec = frames[spec_cols].to_numpy(dtype=float)          # (n_scans, 256)
    dark = frames["dark"].to_numpy(dtype=float)[:, None]    # (n_scans, 1)

    # Absorbance over the full grid, then restrict to the fit window.
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = (spec - dark) / ref[None, :]
        A = -np.log10(np.where(ratio > 0, ratio, np.nan))
    A_w = A[:, in_win]

    n = len(frames)
    no3 = np.full(n, np.nan)
    # Design matrix columns: [1, wl, eps_NO3] -> baseline intercept e, slope f, nitrate c
    base = np.column_stack([np.ones_like(wl_w), wl_w - wl_ref, eps_w])  # centered baseline

    for i in range(n):
        S = salinity[i]; T = temp_c[i]
        if not np.isfinite(S) or not np.isfinite(T):
            continue
        a = A_w[i]
        # bromide (sea-salt) absorbance scaled by salinity and temperature
        a_swa = S * (swa_w + tswa_w * (T - t_cal_swa)) / conc_swa
        a_resid = a - a_swa
        good = np.isfinite(a_resid) & (np.abs(a) < ABSORBANCE_CUTOFF)
        if good.sum() < 5:
            continue
        coef, *_ = np.linalg.lstsq(base[good], a_resid[good], rcond=None)
        no3[i] = coef[2]   # nitrate concentration (uM), path already folded into eps

    out = frames.copy()
    out["no3_tsp"] = no3
    return out


## 6. Build the working SUNA series

Reads the raw frames, attaches CTD temperature and salinity from the step02 merged 1 s product (by
nearest UTC time), and produces the Stage-1 nitrate (`no3_stage1`) according to `STAGE1_MODE`.

In [7]:
def attach_ctd_ts(frames: pd.DataFrame, merged_1s: pd.DataFrame
                  ) -> Tuple[np.ndarray, np.ndarray]:
    """Nearest-time CTD temperature (C) and practical salinity for each SUNA frame."""
    tcol = next((c for c in merged_1s.columns if re.search(r"(t090|tv290|temp|^t\d)", c, re.I)), None)
    scol = next((c for c in merged_1s.columns if re.search(r"sal", c, re.I)), None)
    if tcol is None or scol is None:
        raise ValueError(f"Could not find CTD temp/sal columns in merged file. "
                         f"Columns: {list(merged_1s.columns)}")
    m = merged_1s.dropna(subset=["utc_time"]).copy()
    m["utc_time"] = pd.to_datetime(m["utc_time"], utc=True)
    m = m.sort_values("utc_time")
    idx = pd.DatetimeIndex(m["utc_time"])
    pos = idx.get_indexer(pd.DatetimeIndex(frames["utc_time"]), method="nearest")
    return m[tcol].to_numpy()[pos], m[scol].to_numpy()[pos]


def build_stage1(cast_id: str) -> Tuple[pd.DataFrame, List[str], Dict[str, Any]]:
    # Resolve the raw SUNA file for this cast via the cast->file map (files are
    # named A00000NN.CSV, which do not match the cast id). Fall back to a name
    # glob only if no map entry exists.
    raw_path = None
    suna_map = {}
    if SUNA_MAP_FILE.exists():
        import csv as _csv
        with open(SUNA_MAP_FILE, newline="", encoding="utf-8-sig") as _fh:
            for _r in _csv.DictReader(_fh):
                _c = (_r.get("cast_id") or "").strip()
                _f = (_r.get("suna_file") or "").strip()
                if _c and _f:
                    suna_map[_c] = _f
    if cast_id in suna_map:
        fname = suna_map[cast_id]
        cand = L1_SUNA_ROOT / fname
        raw_path = cand if cand.exists() else next(
            (p for p in L1_SUNA_ROOT.glob("*") if p.name.lower() == fname.lower()), None)
    if raw_path is None:
        raw_path = next(iter(L1_SUNA_ROOT.glob(f"*{cast_id}*.csv")), None) \
            or next(iter(L1_SUNA_ROOT.glob(f"*{cast_id}*.CSV")), None)

    merged_path = L2_SUNA_ROOT / f"{cast_id}_SUNA_1s.csv"
    if raw_path is None or not merged_path.exists():
        raise FileNotFoundError(
            f"Missing raw SUNA (via map) or merged 1s file for {cast_id}. "
            f"Check {SUNA_MAP_FILE.name}.")

    frames, spec_cols = read_suna_frames(raw_path)
    merged = pd.read_csv(merged_path)
    temp_c, sal = attach_ctd_ts(frames, merged)

    if STAGE1_MODE == "tsp":
        frames = tsp_correct(frames, spec_cols, cal, temp_c, sal)
        frames["no3_stage1"] = frames["no3_tsp"]
    else:
        frames["no3_stage1"] = frames["no3_onboard"]

    frames["temp_c"] = temp_c
    frames["salinity"] = sal
    info = {"cast_id": cast_id, "raw_file": str(raw_path), "merged_file": str(merged_path),
            "n_frames": len(frames), "stage1_mode": STAGE1_MODE}
    return frames, spec_cols, info


## 7. Stage 0 — bottle match-up

Pairs each bottle with co-located SUNA scans. If bottle `time_utc` is present, average SUNA within
±`MATCH_TIME_TOL_S`; otherwise match on depth within `MATCH_DEPTH_TOL_M` using the merged 1 s
depth. Applies Zheng's replicate QC: drop replicate pairs differing by more than
`REPLICATE_MAX_DIFF` µM. Returns the paired table that Stages 2–3 regress against.

In [8]:
def load_bottles() -> pd.DataFrame:
    if not BOTTLE_TABLE.exists():
        raise FileNotFoundError(
            f"No bottle table at {BOTTLE_TABLE}. Fill the template from Section 2 first."
        )
    b = pd.read_csv(BOTTLE_TABLE)
    b = b[b["cruise_id"].astype(str).str.strip() == CRUISE_ID].copy()
    b["nitrate_uM"] = pd.to_numeric(b.get("nitrate_uM"), errors="coerce")
    b["depth_m"]    = pd.to_numeric(b.get("depth_m"), errors="coerce")
    if "time_utc" in b.columns:
        b["time_utc"] = pd.to_datetime(b["time_utc"], utc=True, errors="coerce")
    # Replicate QC (Zheng): exclude replicate groups whose spread exceeds the limit.
    if "replicate_id" in b.columns and b["replicate_id"].notna().any():
        keep = []
        for (_, _), grp in b.groupby(["cast_id", "replicate_id"], dropna=False):
            if grp["nitrate_uM"].notna().sum() >= 2 and \
               (grp["nitrate_uM"].max() - grp["nitrate_uM"].min()) > REPLICATE_MAX_DIFF:
                continue
            keep.append(grp)
        b = pd.concat(keep, ignore_index=True) if keep else b.iloc[0:0]
    return b.dropna(subset=["nitrate_uM", "depth_m"])


def match_bottles(frames_by_cast: Dict[str, pd.DataFrame],
                  merged_by_cast: Dict[str, pd.DataFrame],
                  bottles: pd.DataFrame) -> pd.DataFrame:
    recs = []
    for _, row in bottles.iterrows():
        cast = str(row["cast_id"])
        frames = frames_by_cast.get(cast)
        merged = merged_by_cast.get(cast)
        if frames is None or merged is None:
            continue
        depth_col = next((c for c in merged.columns if re.search(r"depS|depth|prDM", c, re.I)), None)

        if pd.notna(row.get("time_utc", pd.NaT)):
            t0 = row["time_utc"]
            sel = frames[(frames["utc_time"] >= t0 - pd.Timedelta(seconds=MATCH_TIME_TOL_S)) &
                         (frames["utc_time"] <= t0 + pd.Timedelta(seconds=MATCH_TIME_TOL_S))]
            suna_no3 = sel["no3_stage1"].mean()
            suna_T   = sel["temp_c"].mean()
            n_used = len(sel)
        elif depth_col is not None:
            # nearest-depth window on the merged product, then map to frames by time
            md = merged.copy()
            md["_dz"] = (md[depth_col] - row["depth_m"]).abs()
            near = md[md["_dz"] <= MATCH_DEPTH_TOL_M]
            if near.empty:
                continue
            tmid = pd.to_datetime(near["utc_time"], utc=True).mean()
            sel = frames[(frames["utc_time"] >= tmid - pd.Timedelta(seconds=MATCH_TIME_TOL_S)) &
                         (frames["utc_time"] <= tmid + pd.Timedelta(seconds=MATCH_TIME_TOL_S))]
            suna_no3 = sel["no3_stage1"].mean() if len(sel) else np.nan
            suna_T   = sel["temp_c"].mean() if len(sel) else np.nan
            n_used = len(sel)
        else:
            continue

        recs.append({
            "cast_id": cast, "station": row.get("station"),
            "depth_m": row["depth_m"], "bottle_no3": row["nitrate_uM"],
            "suna_no3_stage1": suna_no3, "ctd_temp_c": suna_T,
            "n_suna_scans": n_used,
            "residual": suna_no3 - row["nitrate_uM"],
        })
    return pd.DataFrame(recs).dropna(subset=["suna_no3_stage1"])


## 8. Stage 2 — low-nitrate temperature residual

Using match-ups with bottle nitrate < `LOW_NITRATE_THRESHOLD`, regress the residual
(SUNA − bottle) against CTD temperature to get **your own** c1, c2 (Zheng Eq. 3). The slope is
reported so you can compare with Zheng's NES value (c1≈0.057) — a very different slope is a signal
to inspect, not to blindly apply.

In [9]:
def fit_temperature_residual(matchups: pd.DataFrame) -> Dict[str, Any]:
    low = matchups[matchups["bottle_no3"] < LOW_NITRATE_THRESHOLD].dropna(
        subset=["residual", "ctd_temp_c"])
    if len(low) < 3:
        return {"ok": False, "reason": f"only {len(low)} low-nitrate match-ups (<3)",
                "c1": 0.0, "c2": 0.0, "n": len(low)}
    c1, c2 = np.polyfit(low["ctd_temp_c"], low["residual"], 1)
    pred = c1 * low["ctd_temp_c"] + c2
    ss_res = float(((low["residual"] - pred) ** 2).sum())
    ss_tot = float(((low["residual"] - low["residual"].mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {"ok": True, "c1": float(c1), "c2": float(c2), "n": len(low), "r2": r2,
            "zheng_c1_ref": 0.0569}


def apply_temperature_residual(frames: pd.DataFrame, fit: Dict[str, Any]) -> pd.DataFrame:
    out = frames.copy()
    out["no3_stage2"] = out["no3_stage1"]
    if fit.get("ok"):
        low_mask = out["no3_stage1"] < LOW_NITRATE_THRESHOLD
        corr = fit["c1"] * out["temp_c"] + fit["c2"]
        out.loc[low_mask, "no3_stage2"] = out.loc[low_mask, "no3_stage1"] - corr[low_mask]
    return out


## 9. Stage 3 — cruise-specific full-range bias

Regress Stage-2 SUNA against all bottle nitrate to get slope a1, intercept a0, then
`N_QC = (N − a0)/a1` (Zheng Eq. 4). Guards: warns if fewer than `MIN_BOTTLES_FOR_FIT` bottles or if
they do not span the low/middle/high subranges — the single-cruise, few-bottle case Zheng flags as
under-constrained. The NES summer-surface zero-clamp stays off unless explicitly enabled.

In [10]:
def fit_full_range_bias(matchups_stage2: pd.DataFrame) -> Dict[str, Any]:
    d = matchups_stage2.dropna(subset=["suna_no3_stage2", "bottle_no3"])
    n = len(d)
    warnings_list = []
    if n < MIN_BOTTLES_FOR_FIT:
        warnings_list.append(f"only {n} bottles (< {MIN_BOTTLES_FOR_FIT}); fit may be unstable")
    # subrange coverage (Zheng: low 0-2, mid 2-8, high 8-26)
    cov = {"low(0-2)": int(((d["bottle_no3"] >= 0) & (d["bottle_no3"] < 2)).sum()),
           "mid(2-8)": int(((d["bottle_no3"] >= 2) & (d["bottle_no3"] < 8)).sum()),
           "high(8+)": int((d["bottle_no3"] >= 8).sum())}
    missing = [k for k, v in cov.items() if v == 0]
    if missing:
        warnings_list.append(f"no bottles in subrange(s): {missing}; full-range fit not constrained")
    if n < 2:
        return {"ok": False, "a1": 1.0, "a0": 0.0, "n": n, "coverage": cov,
                "warnings": warnings_list + ["too few points to fit"]}
    a1, a0 = np.polyfit(d["bottle_no3"], d["suna_no3_stage2"], 1)
    return {"ok": True, "a1": float(a1), "a0": float(a0), "n": n,
            "coverage": cov, "warnings": warnings_list}


def apply_full_range_bias(frames: pd.DataFrame, fit: Dict[str, Any]) -> pd.DataFrame:
    out = frames.copy()
    a1 = fit["a1"] if fit.get("ok") and fit["a1"] != 0 else 1.0
    a0 = fit["a0"] if fit.get("ok") else 0.0
    out["no3_qc"] = (out["no3_stage2"] - a0) / a1
    return out


## 10. Run the full QC for the cruise

In [11]:
# Discover casts that have both a merged 1s file and raw frames.
cast_ids = sorted({p.name[:-len("_SUNA_1s.csv")]
                   for p in L2_SUNA_ROOT.glob("*_SUNA_1s.csv")})
print("Casts with merged 1s product:", cast_ids)

frames_by_cast: Dict[str, pd.DataFrame] = {}
merged_by_cast: Dict[str, pd.DataFrame] = {}
stage1_info: List[Dict[str, Any]] = []

for cast in cast_ids:
    try:
        fr, _spec, info = build_stage1(cast)
        frames_by_cast[cast] = fr
        merged_by_cast[cast] = pd.read_csv(L2_SUNA_ROOT / f"{cast}_SUNA_1s.csv")
        stage1_info.append(info)
        print(f"{cast}: Stage 1 ({STAGE1_MODE}) on {len(fr)} frames, "
              f"median NO3 = {fr['no3_stage1'].median():.2f} uM")
    except Exception as exc:
        print(f"{cast}: Stage 1 FAILED -> {exc!r}")

display(pd.DataFrame(stage1_info))


Casts with merged 1s product: ['P45_06_CTD_03', 'P45_06_CTD_04', 'P45_06_CTD_05', 'P45_06_CTD_06', 'P45_06_CTD_07', 'P45_06_CTD_08']
P45_06_CTD_03: Stage 1 (onboard) on 1504 frames, median NO3 = -2.69 uM
P45_06_CTD_04: Stage 1 (onboard) on 3196 frames, median NO3 = -1.26 uM
P45_06_CTD_05: Stage 1 (onboard) on 3196 frames, median NO3 = -1.26 uM
P45_06_CTD_06: Stage 1 (onboard) on 1837 frames, median NO3 = -2.95 uM
P45_06_CTD_07: Stage 1 (onboard) on 887 frames, median NO3 = -2.89 uM
P45_06_CTD_08: Stage 1 (onboard) on 1892 frames, median NO3 = -3.00 uM


,cast_id,raw_file,merged_file,n_frames,stage1_mode
0,P45_06_CTD_03,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,1504,onboard
1,P45_06_CTD_04,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,3196,onboard
2,P45_06_CTD_05,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,3196,onboard
3,P45_06_CTD_06,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,1837,onboard
4,P45_06_CTD_07,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,887,onboard
5,P45_06_CTD_08,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,C:\Users\OA_2023-03\Projects\ctd_pipeline_v2\c...,1892,onboard


In [12]:
# Stages 0/2/3 require the bottle table. Run if present; otherwise stop after Stage 1.
if BOTTLE_TABLE.exists():
    bottles = load_bottles()
    print(f"Bottle samples for {CRUISE_ID} after replicate QC: {len(bottles)}")

    matchups = match_bottles(frames_by_cast, merged_by_cast, bottles)
    display(matchups)
    matchups.to_csv(AUDIT_ROOT / "step03_bottle_matchups.csv", index=False, encoding="utf-8-sig")

    t_fit = fit_temperature_residual(matchups)
    print("Stage 2 temperature-residual fit:", t_fit)

    # recompute Stage 2 on the match-ups so Stage 3 regresses corrected SUNA
    mu = matchups.copy()
    if t_fit.get("ok"):
        low = mu["bottle_no3"] < LOW_NITRATE_THRESHOLD
        mu["suna_no3_stage2"] = mu["suna_no3_stage1"]
        mu.loc[low, "suna_no3_stage2"] = (
            mu.loc[low, "suna_no3_stage1"] - (t_fit["c1"] * mu.loc[low, "ctd_temp_c"] + t_fit["c2"]))
    else:
        mu["suna_no3_stage2"] = mu["suna_no3_stage1"]

    b_fit = fit_full_range_bias(mu)
    print("Stage 3 full-range bias fit:", b_fit)
    for w in b_fit.get("warnings", []):
        print("  WARNING:", w)

    # Apply both corrections to every frame and write L3 product per cast.
    qc_summary = []
    for cast, fr in frames_by_cast.items():
        fr2 = apply_temperature_residual(fr, t_fit)
        fr3 = apply_full_range_bias(fr2, b_fit)
        if APPLY_SUMMER_SURFACE_ZERO:
            pass  # NES-specific clamp intentionally not implemented for Gulf of Guinea
        out_cols = ["utc_time", "temp_c", "salinity", "no3_onboard",
                    "no3_stage1", "no3_stage2", "no3_qc"]
        out_cols = [c for c in out_cols if c in fr3.columns]
        out_path = L3_ROOT / f"{cast}_nitrate_QC.csv"
        fr3[out_cols].to_csv(out_path, index=False, encoding="utf-8-sig")
        qc_summary.append({"cast_id": cast, "n": len(fr3),
                           "median_qc_no3": float(np.nanmedian(fr3["no3_qc"])),
                           "output": str(out_path)})
        print(f"{cast}: wrote {out_path.name}")

    qc_df = pd.DataFrame(qc_summary)
    qc_df.to_csv(AUDIT_ROOT / "step03_qc_summary.csv", index=False, encoding="utf-8-sig")
    display(qc_df)
else:
    print("No bottle table yet -> Stage 1 product only. Fill the template (Section 2),")
    print("save as", BOTTLE_TABLE.name, "and rerun to apply Stages 2-3.")


No bottle table yet -> Stage 1 product only. Fill the template (Section 2),
save as P45_06_bottle_nitrate.csv and rerun to apply Stages 2-3.


## 11. Assessment (vs bottle)

RMSE and bias at each correction stage relative to the bottle match-ups, in the style of Zheng's
Table 1 / Fig. 7. Only meaningful once the bottle table is present.

In [13]:
def rmse(a, b):
    d = pd.DataFrame({"a": a, "b": b}).dropna()
    return float(np.sqrt(((d["a"] - d["b"]) ** 2).mean())) if len(d) else np.nan

if BOTTLE_TABLE.exists() and 'matchups' in globals() and not matchups.empty:
    # Recompute stage values at the match-up points for a like-for-like assessment.
    assess = matchups.copy()
    if t_fit.get("ok"):
        low = assess["bottle_no3"] < LOW_NITRATE_THRESHOLD
        assess["s2"] = assess["suna_no3_stage1"]
        assess.loc[low, "s2"] = assess.loc[low, "suna_no3_stage1"] - (
            t_fit["c1"] * assess.loc[low, "ctd_temp_c"] + t_fit["c2"])
    else:
        assess["s2"] = assess["suna_no3_stage1"]
    a1 = b_fit["a1"] if b_fit.get("ok") and b_fit["a1"] != 0 else 1.0
    a0 = b_fit["a0"] if b_fit.get("ok") else 0.0
    assess["qc"] = (assess["s2"] - a0) / a1

    table = pd.DataFrame([
        {"stage": "Stage 1 (TSP/onboard)", "rmse_uM": rmse(assess["suna_no3_stage1"], assess["bottle_no3"])},
        {"stage": "Stage 2 (+temp residual)", "rmse_uM": rmse(assess["s2"], assess["bottle_no3"])},
        {"stage": "Stage 3 (QC, +bottle bias)", "rmse_uM": rmse(assess["qc"], assess["bottle_no3"])},
    ])
    display(table)
    table.to_csv(AUDIT_ROOT / "step03_assessment_rmse.csv", index=False, encoding="utf-8-sig")
    print(f"Match-ups: {len(assess)}   "
          f"Zheng final-QC RMSE reference: 0.34-0.78 uM")
else:
    print("Assessment requires the bottle table and at least one match-up.")


Assessment requires the bottle table and at least one match-up.
